# Denoising Diffusion Probabilistic Models (DDPM)

Neste notebook, exploraremos os **Denoising Diffusion Probabilistic Models (DDPMs)**, ou Modelos Probabilísticos de Difusão com Desruído. Trata-se de uma das famílias mais proeminentes e eficientes de modelos generativos baseados em verossimilhança (*likelihood-based models*). Diferente de fluxos normalizadores e VAEs, os modelos de difusão aproximam a distribuição de probabilidade dos dados através de um processo iterativo estocástico.

Para facilitar a compreensão conceitual e geométrica sem incorrer em alta complexidade computacional, aplicaremos o modelo a um conjunto de dados bidimensional: a distribuição de pontos em formato de dinossauro (*dino*), extraída do clássico conjunto *Datasaurus Dozen*. Mapearemos essa geometria complexa a partir de ruído puramente Gaussiano através de dois processos fundamentais:

1. **Processo Forward (Difusão):** Uma cadeia de Markov que adiciona ruído Gaussiano aos dados de forma incremental ao longo de $T$ passos de tempo, destruindo progressivamente a estrutura geométrica dos dados até que se tornem ruído branco isotrópico.
2. **Processo Reverso (Desruído):** O treinamento de uma rede neural parametrizada para prever e remover o ruído inserido em cada passo temporal, permitindo-nos gerar amostras inéditas partindo de ruído puro.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
from IPython.display import Image, display

In [ ]:
# Configuração de reprodutibilidade e dispositivo
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo ativo: {device}")

## Preparação dos Dados

Para validar a capacidade do modelo de aprender e gerar distribuições com geometrias não lineares arbitrárias, utilizaremos o dataset do dinossauro. Os dados serão carregados do arquivo `data/DatasaurusDozen.tsv`, filtrados para a classe `dino` e normalizados para possuírem média zero e variância unitária. Essa normalização é crucial para manter a estabilidade numérica da rede neural durante o treinamento.

In [ ]:
def load_dino_dataset(n_samples=2000):
    # Carregar o arquivo TSV completo do DatasaurusDozen
    df = pd.read_csv("data/DatasaurusDozen.tsv", sep="\t")
    df = df[df["dataset"] == "dino"]
    
    # Amostrar os pontos aleatoriamente
    rng = np.random.default_rng(42)
    indices = rng.integers(0, len(df), n_samples)
    x = df["x"].iloc[indices].to_numpy()
    y = df["y"].iloc[indices].to_numpy()
    
    # Adicionar um ruído leve para tornar a distribuição mais suave e contínua
    x = x + rng.normal(scale=0.15, size=len(x))
    y = y + rng.normal(scale=0.15, size=len(y))
    
    # Normalizar os dados (centralizar na média e escala unitária)
    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()
    
    points = np.stack((x, y), axis=1)
    return torch.tensor(points, dtype=torch.float32)

# Carregar dados
X_train = load_dino_dataset(n_samples=2000)
dataset = TensorDataset(X_train)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Visualizar a distribuição original (Ground Truth)
plt.figure(figsize=(6, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], s=8, alpha=0.6)
plt.title("Distribuição Real de Dados (x_0)")
plt.xlabel("X")
plt.ylabel("Y")
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.show()

## Processo de Difusão (Forward Process)

O processo forward define uma cadeia de Markov que transiciona a distribuição original $x_0 \sim q(x)$ para um espaço latente Gaussiano. Em cada passo temporal $t \in \{1, \dots, T\}$, adicionamos progressivamente ruído Gaussiano conforme a distribuição de transição:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \eta_t \mathbf{I})$$

Onde $\beta_t \in (0, 1)$ é o schedule de variância que controla a fração de ruído adicionada em cada passo. Graças à reparametrização Gaussiana, podemos expressar a distribuição marginal de $x_t$ diretamente em termos do ponto de partida $x_0$:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) \mathbf{I})$$

Onde definimos $\alpha_t = 1 - \beta_t$ e a variância acumulada $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$. Dessa forma, qualquer passo intermediário $x_t$ pode ser amostrado sem a necessidade de iterar passo a passo:

$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \text{com } \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

Definiremos $T=100$ passos para este dataset 2D, com um schedule linear para as variâncias $\beta_t$.

In [ ]:
# Hiperparâmetros do processo de difusão
n_steps = 100
beta_start = 1e-4
beta_end = 0.01

# Definir schedule linear para betas
betas = torch.linspace(beta_start, beta_end, n_steps).to(device)

# Computar alphas e coeficientes acumulados
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# Coeficientes para a amostragem direta q(x_t | x_0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

def q_sample(x_0, t, noise=None):
    """
    Realiza a amostragem de x_t diretamente a partir de x_0 utilizando a propriedade da variância acumulada.
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    
    # Redimensionar tensores para permitir multiplicação por broadcast
    sqrt_alpha_t = sqrt_alphas_cumprod[t].reshape(-1, 1)
    sqrt_one_minus_alpha_t = sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1)
    
    return sqrt_alpha_t * x_0 + sqrt_one_minus_alpha_t * noise

### Evolução Visual do Processo Forward

Para compreender o impacto do processo forward, podemos visualizar como a geometria nítida do dinossauro se degrada gradualmente até se fundir completamente em uma distribuição normal standard $\mathcal{N}(0, I)$ após $T=100$ passos.

In [ ]:
steps_to_plot = [0, 10, 20, 50, 80, 99]
fig, axs = plt.subplots(1, len(steps_to_plot), figsize=(18, 3.5))

X_train_device = X_train.to(device)

for i, t_val in enumerate(steps_to_plot):
    # Gerar os tempos correspondentes
    t_tensor = torch.full((X_train.shape[0],), t_val, device=device, dtype=torch.long)
    
    # Amostrar os pontos ruidosos
    x_t = q_sample(X_train_device, t_tensor).cpu().numpy()
    
    ax = axs[i]
    ax.scatter(x_t[:, 0], x_t[:, 1], s=3, alpha=0.6, color="#e91e63")
    ax.set_title(f"Passo t={t_val}")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.suptitle("Degradação Gradual da Estrutura (Processo Forward)", y=1.08)
plt.tight_layout()
plt.show()

## Processo Reverso (Reverse Process)

O processo de reversão reconstrói os dados originais a partir do ruído latente Gaussiano. A distribuição reversa completa é dada por uma cadeia de Markov parametrizada por uma rede neural:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 \mathbf{I})$$

Em vez de prever a média $\mu_\theta$ diretamente, provou-se matematicamente que é muito mais estável fazer com que a rede neural estime o ruído $\epsilon$ que foi adicionado em um determinado passo de tempo $t$. A rede neural $\epsilon_\theta(x_t, t)$ recebe a coordenada ruidosa $x_t$ e o passo de tempo $t$, e tenta prever o ruído real $\epsilon \sim \mathcal{N}(0, \mathbf{I})$ que gerou $x_t$.

A função de perda utilizada para o treinamento é o erro quadrático médio (MSE) entre o ruído real adicionado e o ruído previsto:

$$\mathcal{L}_{simple}(\theta) = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

### Arquitetura do Modelo

Como nosso dataset possui apenas duas dimensões ($X \in \mathbb{R}^2$), não necessitamos de uma rede convolucional (como a U-Net, comum em imagens). Em vez disso, usaremos um **MLP (Multi-Layer Perceptron) Condicional**.

Para que a rede saiba com qual nível de ruído está lidando, o passo de tempo $t$ é mapeado para um espaço vetorial de dimensão maior usando *Sinusoidal Position Embeddings*. Esse embedding de tempo é então concatenado com as coordenadas do ponto nas camadas ocultas.

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        self.register_buffer("freq", torch.exp(-math.log(10000) * torch.arange(half) / (half - 1)))

    def forward(self, t):
        x = t.float()[:, None] * self.freq[None, :]
        return torch.cat((x.sin(), x.cos()), dim=-1)

In [ ]:
class Block(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU()
        )
        
    def forward(self, x):
        return self.net(x)

class SimpleDiffusionMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        
        # Mapeamento do embedding temporal
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.GELU()
        )
        
        # O modelo recebe as coordenadas do ponto e o embedding de tempo concatenados
        self.input_layer = nn.Linear(input_dim + time_dim, hidden_dim)
        
        self.block1 = Block(hidden_dim, hidden_dim)
        self.block2 = Block(hidden_dim, hidden_dim)
        
        # Camada de saída (predição de ruído 2D)
        self.output_layer = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        
        # Concatenar a entrada e o embedding de tempo
        x_input = torch.cat([x, t_emb], dim=1)
        
        h = self.input_layer(x_input)
        h = h + self.block1(h)  # Conexão residual simples
        h = h + self.block2(h)
        
        return self.output_layer(h)

In [ ]:
model = SimpleDiffusionMLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print(model)

## Treinamento

O algoritmo de treinamento consiste em:
1. Amostrar um batch de pontos reais $x_0 \sim q(x_0)$.
2. Escolher aleatoriamente um passo temporal $t \sim \text{Uniforme}(\{0, \dots, T-1\})$ para cada ponto do batch.
3. Amostrar um ruído $\epsilon \sim \mathcal{N}(0, \mathbf{I})$.
4. Computar a amostra ruidosa correspondente $x_t = \text{q\_sample}(x_0, t, \epsilon)$.
5. Calcular a perda MSE entre o ruído real $\epsilon$ e o ruído previsto $\epsilon_\theta(x_t, t)$ e atualizar os pesos da rede.

In [ ]:
n_epochs = 2000
losses = []

model.train()

for epoch in range(n_epochs):
    epoch_loss = 0
    for batch in dataloader:
        x0_batch = batch[0].to(device)
        batch_size_current = x0_batch.shape[0]
        
        # 1. Amostrar t uniformemente para cada ponto no batch
        t = torch.randint(0, n_steps, (batch_size_current,), device=device).long()
        
        # 2. Amostrar ruído gaussiano
        noise = torch.randn_like(x0_batch)
        
        # 3. Gerar a amostra ruidosa x_t
        x_t = q_sample(x0_batch, t, noise)
        
        # 4. Prever o ruído
        predicted_noise = model(x_t, t)
        
        # 5. Calcular perda MSE
        loss = nn.MSELoss()(predicted_noise, noise)
        
        # 6. Atualizar os pesos
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 200 == 0:
        print(f"Época {epoch+1}/{n_epochs} | Loss (MSE): {avg_loss:.5f}")

# Plotar a curva de aprendizado
plt.figure(figsize=(7, 4))
plt.plot(losses, color="#9c27b0", linewidth=1.5)
plt.title("Curva de Aprendizado (Perda MSE)")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()

## Geração (Amostragem)

Para gerar novas amostras a partir do modelo treinado, iniciamos com uma distribuição latente de ruído puro $x_T \sim \mathcal{N}(0, \mathbf{I})$ e executamos iterativamente a cadeia de Markov reversa de $t = T-1$ até $t=0$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

Onde $z \sim \mathcal{N}(0, \mathbf{I})$ se $t > 0$ e $z = 0$ quando $t = 0$. A adição de ruído estocástico $z$ em cada passo (controlado pela variância de transição $\sigma_t^2 = \beta_t$) garante a modelagem correta da distribuição.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, shape):
    model.eval()
    device = next(model.parameters()).device
    
    # Iniciar com ruído puro x_T
    x = torch.randn(shape, device=device)
    
    for t in reversed(range(n_steps)):
        t_tensor = torch.full((shape[0],), t, device=device, dtype=torch.long)
        
        # Prever o ruído com o modelo
        eps = model(x, t_tensor)
        
        # Coeficientes correspondentes a t
        a = alphas[t]
        b = betas[t]
        sqrt_oma = sqrt_one_minus_alphas_cumprod[t]
        
        # Calcular a média estimada do passo reverso
        mean = (x - b * eps / sqrt_oma) / torch.sqrt(a)
        
        # Adicionar ruído estocástico se t > 0
        if t == 0:
            x = mean
        else:
            x = mean + torch.sqrt(b) * torch.randn_like(x)
            
    return x

In [ ]:
# Gerar 1000 novos pontos utilizando o processo reverso
generated_points = p_sample_loop(model, (1000, 2)).cpu().numpy()

# Visualizar pontos gerados sobrepostos à distribuição real
plt.figure(figsize=(7, 7))
plt.scatter(X_train[:, 0], X_train[:, 1], c="gray", alpha=0.15, label="Dados Reais (Ground Truth)")
plt.scatter(generated_points[:, 0], generated_points[:, 1], c="#9c27b0", s=6, label="Dados Gerados (DDPM)")
plt.axis("equal")
plt.title("DDPM: Dados Gerados vs Dados Reais")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
@torch.no_grad()
def p_sample_trajectory(model, shape):
    model.eval()
    device = next(model.parameters()).device
    
    # Iniciar com ruído puro
    x = torch.randn(shape, device=device)
    trajectory = [x.cpu().numpy()]
    
    for t in reversed(range(n_steps)):
        t_tensor = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_tensor)
        
        a = alphas[t]
        b = betas[t]
        sqrt_oma = sqrt_one_minus_alphas_cumprod[t]
        
        mean = (x - b * eps / sqrt_oma) / torch.sqrt(a)
        
        if t == 0:
            x = mean
        else:
            x = mean + torch.sqrt(b) * torch.randn_like(x)
            
        trajectory.append(x.cpu().numpy())
        
    return trajectory

In [ ]:
# Obter a trajetória completa
trajectory = p_sample_trajectory(model, (1000, 2))

# Escolher passos temporais para exibir na inferência
steps_plot_reverse = np.linspace(0, n_steps, 6, dtype=int)
fig, axs = plt.subplots(1, len(steps_plot_reverse), figsize=(18, 3.5))

for i, s in enumerate(steps_plot_reverse):
    idx = min(s, len(trajectory) - 1)
    ax = axs[i]
    ax.scatter(trajectory[idx][:, 0], trajectory[idx][:, 1], s=2.5, color="#9c27b0")
    # Mostrar t correspondente (t = T - s)
    t_val = n_steps - idx
    ax.set_title(f"t={t_val}")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.suptitle("Processo Reverso (Geração de Dados: t=100 -> t=0)", y=1.08)
plt.tight_layout()
plt.show()

In [ ]:
def save_ddpm_video(traj, filename="data/ddpm_dino.gif", fps=25):
    frames = []
    
    for idx, data in enumerate(traj):
        # Gerar o frame correspondente
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(data[:, 0], data[:, 1], s=4, color="#9c27b0")
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
        ax.set_title(f"Amostragem DDPM - Passo t={n_steps - idx}")
        ax.grid(True, alpha=0.2)
        
        # Converter a figura do matplotlib em array de pixels
        fig.canvas.draw()
        rgba = np.asarray(fig.canvas.buffer_rgba())
        rgb = rgba[..., :3]
        frames.append(rgb)
        
        plt.tight_layout()
        plt.close(fig)
        
    # Salvar trajetória reversa como animação GIF
    imageio.mimsave(filename, frames, fps=fps)
    print(f"Vídeo/Animação salvo com sucesso em: {filename}")

# Gerar e salvar a animação
save_ddpm_video(trajectory, "data/ddpm_dino.gif", fps=25)

# Exibir a animação gerada no próprio notebook
display(Image(filename="data/ddpm_dino.gif"))